# 10. Навчання на часткових історіях

**Етап 10 з 13** · NASA C-MAPSS FD001 · RUL prediction

**Мета.** Відтворювано оцінити модель за зафіксованим протоколом і зберегти артефакти для подальшого використання.


In [1]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
import copy
import json
import os
import random
import zipfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset


DATA_DIR = Path(os.environ.get("FD001_DIR", "/content/drive/MyDrive/predictive_maintenance"))
OUT_DIR = Path(os.environ.get("FD001_OUT", "outputs"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
WINDOW_SIZE = 80
MAX_EPOCHS = 60
PATIENCE = 10
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
NUM_WORKERS = 2
RUL_CAP_FOR_REPORT = 125

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed()
print("Device:", DEVICE)
print("Data:", DATA_DIR)
print("Output:", OUT_DIR.resolve())


Device: cuda
Data: /content/drive/MyDrive/predictive_maintenance
Output: /content/outputs


## Data loading and group split


In [3]:
COLUMNS = (
    ["unit_id", "cycle", "setting_1", "setting_2", "setting_3"]
    + [f"sensor_{i}" for i in range(1, 22)]
)


def load_fd001(data_dir):
    train = pd.read_csv(data_dir / "train_FD001.txt", sep=r"\s+", header=None, names=COLUMNS)
    test = pd.read_csv(data_dir / "test_FD001.txt", sep=r"\s+", header=None, names=COLUMNS)
    rul = pd.read_csv(data_dir / "RUL_FD001.txt", sep=r"\s+", header=None)[0].to_numpy(float)

    train = train.sort_values(["unit_id", "cycle"]).reset_index(drop=True)
    test = test.sort_values(["unit_id", "cycle"]).reset_index(drop=True)
    train["RUL"] = train.groupby("unit_id")["cycle"].transform("max") - train["cycle"]
    return train, test, rul


train_full, test_full, y_test_true = load_fd001(DATA_DIR)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(splitter.split(train_full, groups=train_full["unit_id"]))
train_raw = train_full.iloc[train_idx].copy()
val_raw = train_full.iloc[val_idx].copy()

print("Train:", train_raw.shape, "engines:", train_raw.unit_id.nunique())
print("Validation:", val_raw.shape, "engines:", val_raw.unit_id.nunique())
print("Official test:", test_full.shape, "engines:", test_full.unit_id.nunique())


Train: (16561, 27) engines: 80
Validation: (4070, 27) engines: 20
Official test: (13096, 26) engines: 100


## Feature engineering

The detector is fitted only on the 80 training engines. The first 20 cycles
have no degradation score, matching the application pipeline.


In [4]:
DETECTOR_DIRECTIONS = {
    "sensor_4": 1,
    "sensor_7": -1,
    "sensor_11": 1,
    "sensor_12": -1,
    "sensor_15": 1,
    "sensor_20": -1,
    "sensor_21": -1,
}
BASELINE_CYCLES = 20
DETECTOR_WINDOW = 10

BASE_FEATURES = (
    ["cycle", "setting_1", "setting_2", "setting_3"]
    + [f"sensor_{i}" for i in range(1, 22)]
)

MODEL_CONFIGS = {
    "lstm": {
        "flag_column": "degradation_warmup",
        "flag_mode": "warmup",
    },
    "cnn": {
        "flag_column": "degradation_available",
        "flag_mode": "available",
    },
}


class DegradationScore:
    def __init__(self, baseline_cycles=BASELINE_CYCLES, window=DETECTOR_WINDOW):
        self.baseline_cycles = baseline_cycles
        self.window = window
        self.sensors = list(DETECTOR_DIRECTIONS)
        self.direction = pd.Series(DETECTOR_DIRECTIONS)
        self.scale_ = None

    def fit(self, train_df):
        residuals = []
        for _, engine in train_df.sort_values("cycle").groupby("unit_id", sort=False):
            calibration = engine[self.sensors].iloc[: self.baseline_cycles]
            residuals.append(calibration.sub(calibration.median()))
        residuals = pd.concat(residuals, ignore_index=True)
        self.scale_ = residuals.abs().median().mul(1.4826).clip(lower=1e-6)
        return self

    def _engine_score(self, engine):
        values = engine[self.sensors]
        baseline = values.iloc[: self.baseline_cycles].median()
        deviation = values.sub(baseline).div(self.scale_)
        directional = deviation.mul(self.direction).clip(lower=0)
        return directional.mean(axis=1).rolling(self.window, min_periods=1).mean()

    def transform(self, df):
        result = pd.Series(np.nan, index=df.index, dtype=float, name="degradation_score")
        for _, engine in df.groupby("unit_id", sort=False):
            engine = engine.sort_values("cycle")
            score = self._engine_score(engine).to_numpy(copy=True)
            score[: self.baseline_cycles] = np.nan
            result.loc[engine.index] = score
        return result

    def to_dict(self):
        return {
            "version": 1,
            "baseline_cycles": self.baseline_cycles,
            "window": self.window,
            "directions": DETECTOR_DIRECTIONS,
            "scale": {name: float(value) for name, value in self.scale_.items()},
        }


def build_features(df, detector):
    out = df.copy()
    out["degradation_score"] = detector.transform(out)
    return out


def add_flag(df, mode, column):
    out = df.copy()
    missing = out["degradation_score"].isna()
    if mode == "warmup":
        out[column] = missing.astype("float32")
    elif mode == "available":
        out[column] = (~missing).astype("float32")
    else:
        raise ValueError(f"Unknown flag mode: {mode}")
    return out


detector = DegradationScore().fit(train_raw)
train_features = build_features(train_raw, detector)
val_features = build_features(val_raw, detector)
test_features = build_features(test_full, detector)

print("Detector scale:")
display(detector.scale_.to_frame("scale"))


Detector scale:


,scale
sensor_4,3.776924
sensor_7,0.392889
sensor_11,0.096369
sensor_12,0.285400
sensor_15,0.018755
sensor_20,0.088956
sensor_21,0.059934


## Model-specific scaling


In [5]:
def prepare_model_frames(train_df, val_df, test_df, model_name):
    config = MODEL_CONFIGS[model_name]
    flag = config["flag_column"]
    columns = BASE_FEATURES + ["degradation_score", flag]

    train_out = add_flag(train_df, config["flag_mode"], flag)
    val_out = add_flag(val_df, config["flag_mode"], flag)
    test_out = add_flag(test_df, config["flag_mode"], flag)

    scaler = StandardScaler().fit(train_out[columns].fillna(0).to_numpy(float))
    for frame in (train_out, val_out, test_out):
        frame[columns] = scaler.transform(frame[columns].fillna(0).to_numpy(float))

    return train_out, val_out, test_out, columns, scaler


prepared = {}
for model_name in MODEL_CONFIGS:
    prepared[model_name] = prepare_model_frames(
        train_features,
        val_features,
        test_features,
        model_name,
    )
    print(model_name.upper(), "input features:", len(prepared[model_name][3]))


LSTM input features: 27
CNN input features: 27


## Prefix datasets

Кожен цикл стає одним навчальним прикладом. Префікси, коротші за 80 циклів, доповнюються зліва першим перетвореним рядком, отриманим від рушія.


In [6]:
class PrefixWindowDataset(Dataset):
    def __init__(self, df, columns, window_size=WINDOW_SIZE):
        self.window_size = window_size
        self.engines = []
        self.samples = []

        for engine_index, (_, engine) in enumerate(
            df.sort_values("cycle").groupby("unit_id", sort=False)
        ):
            features = engine[columns].to_numpy(np.float32)
            targets = engine["RUL"].to_numpy(np.float32)
            self.engines.append((features, targets))
            self.samples.extend((engine_index, end) for end in range(1, len(engine) + 1))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        engine_index, end = self.samples[index]
        features, targets = self.engines[engine_index]
        start = max(0, end - self.window_size)
        window = features[start:end]
        real_length = len(window)

        if real_length < self.window_size:
            padding = np.repeat(features[:1], self.window_size - real_length, axis=0)
            window = np.vstack([padding, window])

        return (
            torch.from_numpy(window),
            torch.tensor(targets[end - 1], dtype=torch.float32),
            torch.tensor(real_length, dtype=torch.int64),
        )


def make_last_windows(df, columns, window_size=WINDOW_SIZE):
    windows = []
    unit_ids = []
    lengths = []

    for unit_id, engine in df.sort_values("cycle").groupby("unit_id", sort=True):
        features = engine[columns].to_numpy(np.float32)
        real_length = min(len(features), window_size)
        if len(features) < window_size:
            padding = np.repeat(features[:1], window_size - len(features), axis=0)
            features = np.vstack([padding, features])
        windows.append(features[-window_size:])
        unit_ids.append(unit_id)
        lengths.append(real_length)

    return (
        np.asarray(unit_ids),
        np.asarray(windows, dtype=np.float32),
        np.asarray(lengths),
    )


datasets = {}
for model_name, (train_df, val_df, _, columns, _) in prepared.items():
    datasets[model_name] = {
        "train": PrefixWindowDataset(train_df, columns),
        "val": PrefixWindowDataset(val_df, columns),
    }
    print(
        model_name.upper(),
        "train prefixes:", len(datasets[model_name]["train"]),
        "validation prefixes:", len(datasets[model_name]["val"]),
    )


LSTM train prefixes: 16561 validation prefixes: 4070
CNN train prefixes: 16561 validation prefixes: 4070


## Models and training


In [7]:
class RULLSTM(nn.Module):
    def __init__(self, in_features, hidden_size=64):
        super().__init__()
        self.lstm = nn.LSTM(in_features, hidden_size, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (hidden, _) = self.lstm(x)
        return self.fc(hidden[-1]).squeeze(-1)


class RULCNN(nn.Module):
    def __init__(self, in_features, window_size=WINDOW_SIZE, dropout=0.3):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv1d(in_features, 32, kernel_size=3),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=3),
            nn.ReLU(),
        )
        flattened = 64 * (window_size - 4)
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.fc_layers(self.conv_layers(x.transpose(1, 2))).squeeze(-1)


def regression_metrics(y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred), 0, None)
    y_true = np.asarray(y_true)
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
    }


def evaluate_loader(model, loader):
    model.eval()
    targets, predictions, lengths = [], [], []
    with torch.inference_mode():
        for x_batch, y_batch, length_batch in loader:
            prediction = model(x_batch.to(DEVICE, non_blocking=True))
            targets.append(y_batch.numpy())
            predictions.append(prediction.cpu().numpy())
            lengths.append(length_batch.numpy())
    return np.concatenate(targets), np.concatenate(predictions), np.concatenate(lengths)


def metrics_by_history_length(y_true, y_pred, lengths):
    buckets = {
        "all": np.ones(len(lengths), dtype=bool),
        "1-19": lengths < 20,
        "20-79": (lengths >= 20) & (lengths < WINDOW_SIZE),
        "80+": lengths >= WINDOW_SIZE,
    }
    rows = []
    for name, mask in buckets.items():
        if mask.any():
            rows.append({"history": name, "samples": int(mask.sum()), **regression_metrics(y_true[mask], y_pred[mask])})
    return pd.DataFrame(rows)


def train_model(model, train_dataset, val_dataset, model_name):
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS > 0,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE * 2,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=NUM_WORKERS > 0,
    )

    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.MSELoss()

    best_mae = float("inf")
    best_epoch = 0
    best_state = None
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total_loss = 0.0

        for x_batch, y_batch, _ in train_loader:
            x_batch = x_batch.to(DEVICE, non_blocking=True)
            y_batch = y_batch.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            prediction = model(x_batch)
            loss = loss_fn(prediction, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            total_loss += loss.item() * len(x_batch)

        y_val, val_prediction, val_lengths = evaluate_loader(model, val_loader)
        val_mae = mean_absolute_error(y_val, np.clip(val_prediction, 0, None))
        train_mse = total_loss / len(train_dataset)
        history.append({"epoch": epoch, "train_mse": train_mse, "val_mae": val_mae})
        print(f"{model_name.upper()} epoch {epoch:02d}: train MSE={train_mse:.3f}, val MAE={val_mae:.3f}")

        if val_mae < best_mae:
            best_mae = val_mae
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                print(f"{model_name.upper()} early stopping")
                break

    model.load_state_dict(best_state)
    y_val, val_prediction, val_lengths = evaluate_loader(model, val_loader)
    bucket_metrics = metrics_by_history_length(y_val, val_prediction, val_lengths)
    print(f"{model_name.upper()} best epoch: {best_epoch}, validation MAE: {best_mae:.3f}")
    display(bucket_metrics.round(3))

    return model, {
        "best_epoch": best_epoch,
        "best_val_mae": float(best_mae),
        "history": history,
        "validation_by_history": bucket_metrics.to_dict(orient="records"),
    }


models = {}
training_reports = {}

for model_name in ("lstm", "cnn"):
    set_seed()
    input_size = len(prepared[model_name][3])
    model = RULLSTM(input_size) if model_name == "lstm" else RULCNN(input_size)
    models[model_name], training_reports[model_name] = train_model(
        model,
        datasets[model_name]["train"],
        datasets[model_name]["val"],
        model_name,
    )


LSTM epoch 01: train MSE=15296.397, val MAE=94.442
LSTM epoch 02: train MSE=13328.664, val MAE=86.463
LSTM epoch 03: train MSE=11772.894, val MAE=78.859
LSTM epoch 04: train MSE=10381.936, val MAE=71.788
LSTM epoch 05: train MSE=9121.358, val MAE=65.205
LSTM epoch 06: train MSE=7991.468, val MAE=59.470
LSTM epoch 07: train MSE=7005.447, val MAE=53.726
LSTM epoch 08: train MSE=6149.282, val MAE=49.104
LSTM epoch 09: train MSE=5443.096, val MAE=45.288
LSTM epoch 10: train MSE=4890.403, val MAE=41.955
LSTM epoch 11: train MSE=4402.263, val MAE=38.944
LSTM epoch 12: train MSE=3950.070, val MAE=36.156
LSTM epoch 13: train MSE=3583.369, val MAE=34.058
LSTM epoch 14: train MSE=3300.374, val MAE=32.587
LSTM epoch 15: train MSE=3040.907, val MAE=30.610
LSTM epoch 16: train MSE=2797.073, val MAE=28.981
LSTM epoch 17: train MSE=2595.798, val MAE=27.464
LSTM epoch 18: train MSE=2435.396, val MAE=26.464
LSTM epoch 19: train MSE=2300.199, val MAE=26.076
LSTM epoch 20: train MSE=2178.052, val MAE=25.

,history,samples,MAE,RMSE,R2
0,all,4070,18.622,29.148,0.803
1,1-19,380,42.261,52.943,-0.597
2,20-79,1200,32.343,39.967,0.211
3,80+,2490,8.402,13.825,0.910


CNN epoch 01: train MSE=3318.986, val MAE=25.036
CNN epoch 02: train MSE=1184.725, val MAE=23.723
CNN epoch 03: train MSE=896.710, val MAE=26.010
CNN epoch 04: train MSE=699.856, val MAE=27.351
CNN epoch 05: train MSE=599.383, val MAE=27.057
CNN epoch 06: train MSE=526.873, val MAE=27.836
CNN epoch 07: train MSE=492.636, val MAE=28.554
CNN epoch 08: train MSE=458.339, val MAE=27.046
CNN epoch 09: train MSE=417.098, val MAE=26.593
CNN epoch 10: train MSE=409.058, val MAE=27.735
CNN epoch 11: train MSE=390.128, val MAE=25.649
CNN epoch 12: train MSE=376.193, val MAE=28.125
CNN early stopping
CNN best epoch: 2, validation MAE: 23.723


,history,samples,MAE,RMSE,R2
0,all,4070,23.723,33.823,0.735
1,1-19,380,52.012,62.213,-1.205
2,20-79,1200,35.740,43.923,0.047
3,80+,2490,13.615,18.695,0.836


## Official test evaluation


In [8]:
test_rows = []
test_predictions = pd.DataFrame({"unit_id": np.sort(test_full.unit_id.unique()), "RUL_true": y_test_true})

for model_name, model in models.items():
    _, _, test_df, columns, _ = prepared[model_name]
    unit_ids, x_test, history_lengths = make_last_windows(test_df, columns)

    model.eval()
    with torch.inference_mode():
        prediction = model(torch.from_numpy(x_test).to(DEVICE)).cpu().numpy()
    prediction = np.clip(prediction, 0, None)

    raw_metrics = regression_metrics(y_test_true, prediction)
    capped_metrics = regression_metrics(
        np.minimum(y_test_true, RUL_CAP_FOR_REPORT),
        np.minimum(prediction, RUL_CAP_FOR_REPORT),
    )

    test_rows.append({"model": model_name.upper(), "protocol": "raw", **raw_metrics})
    test_rows.append({"model": model_name.upper(), "protocol": f"capped_{RUL_CAP_FOR_REPORT}", **capped_metrics})
    test_predictions[model_name.upper()] = prediction
    test_predictions[f"{model_name.upper()}_history_cycles"] = history_lengths

test_metrics = pd.DataFrame(test_rows)
display(test_metrics.round(3))


,model,protocol,MAE,RMSE,R2
0,LSTM,raw,13.118,20.376,0.760
1,LSTM,capped_125,8.093,12.298,0.906
2,CNN,raw,17.661,25.757,0.616
3,CNN,capped_125,11.318,16.043,0.840


## Export weights and preprocessing artifacts


In [9]:
model_paths = {
    "lstm": OUT_DIR / "best_lstm_partial_w80.pt",
    "cnn": OUT_DIR / "best_cnn_partial_w80.pt",
}

for model_name, model in models.items():
    cpu_state = {name: tensor.detach().cpu() for name, tensor in model.state_dict().items()}
    torch.save(cpu_state, model_paths[model_name])

joblib.dump(prepared["lstm"][4], OUT_DIR / "scaler_lstm_fd001.pkl")
joblib.dump(prepared["cnn"][4], OUT_DIR / "scaler_cnn_fd001.pkl")

(OUT_DIR / "degradation_detector.json").write_text(
    json.dumps(detector.to_dict(), indent=2, ensure_ascii=False),
    encoding="utf-8",
)

feature_metadata = {}
for model_name in MODEL_CONFIGS:
    columns = prepared[model_name][3]
    feature_metadata[model_name] = {
        "features": columns,
        "n_features": len(columns),
        "flag_mode": MODEL_CONFIGS[model_name]["flag_mode"],
        "window_size": WINDOW_SIZE,
        "trained_on_partial_histories": True,
        "minimum_history_cycles": 1,
    }

(OUT_DIR / "feature_metadata.json").write_text(
    json.dumps(feature_metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
(OUT_DIR / "training_report.json").write_text(
    json.dumps(training_reports, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
(OUT_DIR / "split_metadata.json").write_text(
    json.dumps(
        {
            "seed": SEED,
            "train_engine_ids": sorted(int(x) for x in train_raw.unit_id.unique()),
            "validation_engine_ids": sorted(int(x) for x in val_raw.unit_id.unique()),
        },
        indent=2,
    ),
    encoding="utf-8",
)

test_metrics.to_csv(OUT_DIR / "official_test_metrics.csv", index=False)
test_predictions.to_csv(OUT_DIR / "official_test_predictions.csv", index=False)

bundle_path = OUT_DIR / "fd001_partial_history_models.zip"
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT_DIR.iterdir()):
        if path.is_file() and path != bundle_path:
            archive.write(path, arcname=path.name)

print("Saved files:")
for path in sorted(OUT_DIR.iterdir()):
    print(" -", path)


Saved files:
 - outputs/best_cnn_partial_w80.pt
 - outputs/best_lstm_partial_w80.pt
 - outputs/degradation_detector.json
 - outputs/fd001_partial_history_models.zip
 - outputs/feature_metadata.json
 - outputs/official_test_metrics.csv
 - outputs/official_test_predictions.csv
 - outputs/scaler_cnn_fd001.pkl
 - outputs/scaler_lstm_fd001.pkl
 - outputs/split_metadata.json
 - outputs/training_report.json


## Download

Running this cell downloads both `.pt` files and one ZIP containing the
models, scalers, detector, metadata, metrics, and predictions.


In [10]:
from google.colab import files

files.download(str(model_paths["lstm"]))
files.download(str(model_paths["cnn"]))
files.download(str(bundle_path))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>